    
    
# ****How to build Logistic Regression from Scratch****


## **Key Concepts**



- **Linear Combinations**
- **Sigmoid Activation**
- **Probability**
- **Gradient Descent**


In [918]:
#Important Libraries To Know:

#Numpy: fast math and array operations
import numpy as np

#Pandas: data cleaning and manipulation
import pandas as pd

## **What is $\mathbf{x}$?**


Before diving into the core concepts of logistic regression, we need to figure out what $\mathbf{x}$ is in ML.

**Definition:** a single row of data.

For example, one passenger (a row) with $n$ features:

$$
\mathbf{x}^{(i)} =
\begin{bmatrix}
x^{(i)}_1 & x^{(i)}_2 & \cdots & x^{(i)}_n
\end{bmatrix}
=
\begin{bmatrix}
\text{Age} & \text{Pclass} & \text{Fare} & \cdots
\end{bmatrix}
$$

It is a vector holding the values (value 1, value 2, ..., value $n$) of row $i$.


A full dataset stacks $m$ rows on top of each other into a matrix $X$:

$$
X =
\begin{bmatrix}
\mathbf{x}^{(1)} \\
\mathbf{x}^{(2)} \\
\vdots \\
\mathbf{x}^{(i)} \\
\vdots \\
\mathbf{x}^{(m)}
\end{bmatrix}
\in \mathbb{R}^{m \times n}
$$




**Example:**

$$
X =
\begin{bmatrix}
x^{(1)}_1 & x^{(1)}_2 & x^{(1)}_3 \\
x^{(2)}_1 & x^{(2)}_2 & x^{(2)}_3 \\
x^{(3)}_1 & x^{(3)}_2 & x^{(3)}_3
\end{bmatrix}
=
\begin{bmatrix}
-6.6 & 3.5 & 2.0 \\
1.2 & -0.8 & 4.1 \\
0.5 & 2.3 & -1.7
\end{bmatrix}
$$

There are $3$ Features and $3$ Rows meaning: 

- $m$ is equal to 3 
- $n$ is equal to 3



In [919]:
#x explanation

#simulating data containing 3 features and 1 row, shape (1, 3)
data = pd.DataFrame({
    'Feature1': [-6.6, 1.2, 0.5],
    'Feature2': [3.5, -0.8, 4.1],
    'Feature3': [2.0, 4.1, -1.7]
}, index=[1, 2, 3])

#x is a vector containing a row of data
x = data.to_numpy()

#printing the results
print(data)
print('\nThe "x" for the first row')

#we print out x[0] instead of x[1]
#because x[0] is the first element
#while x[1] is the second
print(f'\n x: {x[0]}')

   Feature1  Feature2  Feature3
1      -6.6       3.5       2.0
2       1.2      -0.8       4.1
3       0.5       4.1      -1.7

The "x" for the first row

 x: [-6.6  3.5  2. ]


## **Linear Combination**


Similar to linear regression, logistic regression starts by taking x vectors, multiplying all the values in those vectors by weights, adding them up, and then adding a bias. The formula:

$$z = \sum_{j=1}^{n} w_j x_j + b$$

Simplified with linear algebra notation:

$$\mathbf{z} = X\mathbf{w} + b$$

In [920]:
#Linear Combination Example

#data simulation
n_samples = 5

#feature lists
np.random.seed(42)
f1 = np.random.randint(1, 5, n_samples)
f2 = np.random.randint(1, 5, n_samples)
f3 = np.random.randint(1, 5, n_samples)

#organize data
df = pd.DataFrame({
    'Feature 1': f1,
    'Feature 2': f2,
    'Feature 3': f3
})

#transform data
data = df.to_numpy(dtype=float)

#example weights for the 3 features (how much each feature matters)
weights = [[0.1], [0.05], [0.03]]

#example bias -0.05
bias = -0.05

#linear combination parameters X -> data, w -> weights, b -> bias
def compute_z(X, w, b):

    #For each row, multiply each value in the x vector by the weight
    #add them up, and add the bias to the total.
    
    return X @ w + b
    #Note: There is a "z" per row



#Add the "z" values to the data frame
df['scores(z)'] = compute_z(data, weights, bias)

#print the result
print(df)


   Feature 1  Feature 2  Feature 3  scores(z)
0          3          4          3       0.54
1          4          1          3       0.49
2          1          1          3       0.19
3          3          3          3       0.49
4          3          2          4       0.47


## **Sigmoid Activation**


This Formula is the **"magic"** behind logistic regression
It can take any real number and transform it into a **probability**

$\sigma(z)=\frac{1}{1+e^{-z}}$


In [921]:
# Scripting simple sigmoid calculator

#setup arrays
z_array = []
y_preds_array = []

#sigmoid activation function
def sigmoid(z): 
    #calculate predicted probability (in decimal form)
    return 1 / (1 + np.exp(-z))

#loop through each example
for iteration in range(1, 11):
    #z = floats 0.1 to 1
    z = iteration/10

    #apply sigmoid function
    y_prediction = sigmoid(z)

    #append to arrays
    z_array.append(z)
    y_preds_array.append(y_prediction)

#setup dataframe to organize data
df = pd.DataFrame({
    'Scores(z)': z_array,
    #                              convert decimals into percents
    'Predicted Probabilities(y)': [f'{(decimal * 100):.3}%' for decimal in y_preds_array]
})

#printing the results
print(df)




   Scores(z) Predicted Probabilities(y)
0        0.1                      52.5%
1        0.2                      55.0%
2        0.3                      57.4%
3        0.4                      59.9%
4        0.5                      62.2%
5        0.6                      64.6%
6        0.7                      66.8%
7        0.8                      69.0%
8        0.9                      71.1%
9        1.0                      73.1%


  ## **Probability Based Predictions**

$$\hat{y} = \sigma(z)$$

$$
\text{prediction} =
\begin{cases}
1 & \text{if } \hat{y} \ge \tau \\
0 & \text{if } \hat{y} < \tau
\end{cases}
$$

This expression above represents the core theory of logistic regression.
After implementing sigmoid function, you take the probability and make a prediction based
on a threshold

For example: if $z$ = $0.5$, and the threshold ($\tau$) = $0.5$ (50%)

1) Predict a percentage: $\hat{y}$ via $\sigma(z)$
2) If $\hat{y}$ (around 62%) is larger or equal than the threshold ($\tau$), then predict $1$
3) If $\hat{y}$ is less than $\tau$ predict $0$ 
4) Since 62% $>$ 50% then the model predicts $1$ 

In [922]:
#Simple Probability Prediction Calculator

#number of samples and set random seed
n_samples = 10
np.random.seed(42)

#setting threshold of 50%
thresh = 0.5

#random set of scores
z = np.random.random(size=n_samples)

#z is the probability created by the sigmoid function 
y_probabilities = sigmoid(z)

#predict 1 if z >= to 50% else predict 0
predictions = np.where(z >= thresh, 1, 0)

#use dataframe for organization
df = pd.DataFrame({
    'Scores(z)': z,
    'Probabilities': [f'{(prob * 100):0.1f}%' for prob in y_probabilities],
    'Predictions': predictions
})
print(df)

   Scores(z) Probabilities  Predictions
0   0.374540         59.3%            0
1   0.950714         72.1%            1
2   0.731994         67.5%            1
3   0.598658         64.5%            1
4   0.156019         53.9%            0
5   0.155995         53.9%            0
6   0.058084         51.5%            0
7   0.866176         70.4%            1
8   0.601115         64.6%            1
9   0.708073         67.0%            1


## **Loss Function: Binary Cross-Entropy Loss** ###



Having a "s" shape curve and a way to predict values, we now need to be able to measure how accurate those predictions are. 

Binary Cross-Entropy Loss or "log loss":

$$J(\mathbf{w}, b) = -\frac{1}{m} \sum_{i=1}^{m} \Big[ y_i \log(\hat{y}_i) + (1 - y_i)\log(1 - \hat{y}_i) \Big]$$

Lets break down this expression

1) Variables:

    "$m$" is equal to the total of the training samples

    "$i$" is the index of the current index    

    "$y_i$" is the true value (1 or 0) of the current index

    "$\hat{y}_i$" is the predicted PROBABILITY of the current index


2) Sigma Notation:

    $-\frac{1}{m} \sum_{i=1}^{m}$ represents the mean of all training losses


3) Overview: This equation calculates the average loss across all training examples by penalizing the model's predicted probabilities based on how far they diverge from the true binary labels.


Log Loss equation simplified:
$$
\ell_i =
\begin{cases}
-\log(\hat{y}_i) & \text{if } y_i = 1 \\
-\log(1 - \hat{y}_i) & \text{if } y_i = 0
\end{cases}
$$

$$J = \frac{1}{m}\sum_{i=1}^{m} \ell_i$$

This expression is a simplified version of the Binary Cross Entropy Loss Function

- If the true value of the current index is $1$, then the loss function is $-\log(\hat{y}_i)$
- Else if the true value of the current index is $0$, then the function is $-\log(1 - \hat{y}_i)$
- Then take the mean of all of the losses, $J$, as the result of the function 

In [923]:
#Loss Calculator

#set seed
np.random.seed(1)

#make true value (1 or 0) array "y" 
y = np.random.choice((1, 0), n_samples)

#make a list of predicted probabilities "^y"
probabilities = np.random.choice((0.2, 0.4, 0.6, 0.8), n_samples)

#calculating loss:

#this is a simplified implementation of Log Loss function
#if y is equal to 1 then use -log(yi) else if 0 use -log(1 - yi)
loss_array = np.where(y == 1,-np.log(probabilities), -np.log(1 - probabilities))

# Calculating the mean of the individual losses
# Binary Cross-Entropy Loss result
loss = loss_array.mean()

#dataframe to display organized data with the y, ^y, and individual losses
df = pd.DataFrame({
    ' y True Values': y,
    ' ^y Predicted %': [f'{p * 100}%' for p in probabilities],
    ' j(y) losses': loss_array
})

#displaying results
print(df)
print(f"\n Binary Cross Entropy Loss: {loss}")

   y True Values ^y Predicted %  j(y) losses
0              0          20.0%     0.223144
1              0          40.0%     0.510826
2              1          20.0%     1.609438
3              1          80.0%     0.223144
4              0          40.0%     0.510826
5              0          20.0%     0.223144
6              0          60.0%     0.916291
7              0          40.0%     0.510826
8              0          60.0%     0.916291
9              1          20.0%     1.609438

 Binary Cross Entropy Loss: 0.7253364813857112
